# 下载所有模型和数据集

本脚本下载实验所需的全部资源到 `./resources` 文件夹：
- Qwen3-0.6B, Qwen3-1.7B, Qwen3-4B 模型
- GSM8K 数据集

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import snapshot_download

load_dotenv()
if os.getenv("HF_ENDPOINT"):
    os.environ["HF_ENDPOINT"] = os.getenv("HF_ENDPOINT")
else:
    os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

RESOURCES_DIR = "./resources"
os.makedirs(RESOURCES_DIR, exist_ok=True)

print(f"资源将下载到: {os.path.abspath(RESOURCES_DIR)}\n")

## 1. 下载 Qwen3 模型系列

In [ ]:
models = [
    {"id": "Qwen/Qwen3-0.6B", "local_dir": os.path.join(RESOURCES_DIR, "Qwen3-0.6B")},
    {"id": "Qwen/Qwen3-1.7B", "local_dir": os.path.join(RESOURCES_DIR, "Qwen3-1.7B")},
    {"id": "Qwen/Qwen3-4B", "local_dir": os.path.join(RESOURCES_DIR, "Qwen3-4B")},
]

for model in models:
    print(f"下载模型 {model['id']} ...")
    snapshot_download(
        repo_id=model['id'],
        local_dir=model['local_dir']
    )
    print(f"✅ {model['id']} 下载完成！\n")

print("所有模型下载完成！")

## 2. 下载 GSM8K 数据集

In [ ]:
dataset_id = "openai/gsm8k"
dataset_local_dir = os.path.join(RESOURCES_DIR, "gsm8k")

print(f"下载数据集 {dataset_id} ...")
snapshot_download(
    repo_id=dataset_id,
    repo_type="dataset",
    local_dir=dataset_local_dir
)
print(f"✅ 数据集下载完成！")

## 3. 验证下载

In [ ]:
print("\n" + "="*60)
print("下载验证")
print("="*60)

print("\n模型文件:")
for model in models:
    local_dir = model['local_dir']
    if os.path.exists(local_dir):
        files = os.listdir(local_dir)
        safetensors = [f for f in files if f.endswith('.safetensors')]
        print(f"  ✅ {os.path.basename(local_dir)}: {len(safetensors)} 个模型文件")
    else:
        print(f"  ❌ {os.path.basename(local_dir)}: 未找到")

print("\n数据集文件:")
if os.path.exists(dataset_local_dir):
    main_dir = os.path.join(dataset_local_dir, "main")
    if os.path.exists(main_dir):
        files = os.listdir(main_dir)
        print(f"  ✅ gsm8k/main: {files}")
    else:
        print(f"  ❌ {main_dir}: 未找到")
else:
    print(f"  ❌ {dataset_local_dir}: 未找到")

print("\n" + "="*60)

In [ ]:
from datasets import load_dataset

print("\n测试加载数据集...")
data = load_dataset(dataset_local_dir, 'main')
print(f"训练集: {len(data['train'])} 条")
print(f"测试集: {len(data['test'])} 条")
print(f"\n示例数据:")
print(f"  问题: {data['train'][0]['question']}")
print(f"  答案: {data['train'][0]['answer'][:100]}...")

## 4. 显示目录结构和磁盘占用

In [ ]:
import subprocess

print(f"\n目录结构: {os.path.abspath(RESOURCES_DIR)}")
print("-" * 60)

if os.path.exists(RESOURCES_DIR):
    for item in sorted(os.listdir(RESOURCES_DIR)):
        item_path = os.path.join(RESOURCES_DIR, item)
        if os.path.isdir(item_path):
            result = subprocess.run(['du', '-sh', item_path], capture_output=True, text=True)
            size = result.stdout.split()[0]
            print(f"  📁 {item}/ ({size})")
        else:
            print(f"  📄 {item}")

result = subprocess.run(['du', '-sh', RESOURCES_DIR], capture_output=True, text=True)
total_size = result.stdout.split()[0]
print(f"\n总占用: {total_size}")